In [2]:
import pandas as pd

df = pd.read_csv("advanced.csv")
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'advanced.csv'

In [ ]:
print(df.shape)
print(df.columns.tolist())

df = df[df["Player"] != "Player"]
print(df.shape)


In [ ]:
# mark the combined rows for traded players
df["is_total"] = df["Team"].str.contains("TM|TOT", na=False)

# put combined rows first, then keep one row per player
df = df.sort_values("is_total", ascending=False)
df = df.drop_duplicates(subset="Player-additional", keep="first")
df = df.drop(columns="is_total")

# turn the stat columns into real numbers
for c in ["Age", "G", "MP", "PER", "WS", "BPM", "VORP"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df.shape)
df[["Player", "Team", "Age", "MP", "WS", "VORP"]].sort_values("WS", ascending=False).head(10)


In [ ]:
# keep players with real minutes
df = df[df["MP"] >= 500]
print(df.shape)

# first chart: minutes vs win shares
import matplotlib.pyplot as plt

plt.scatter(df["MP"], df["WS"], alpha=0.6)
plt.xlabel("Minutes played")
plt.ylabel("Win Shares")
plt.title("Minutes vs. Win Shares, 2025-26")
plt.show()


In [ ]:
sal = pd.read_csv("salaries.csv")
print(sal.shape)
print(sal.columns.tolist())
sal.head(10)


In [ ]:
import unicodedata, re

# skip the title row so RK / NAME / TEAM / SALARY become the headers
sal = pd.read_csv("salaries.csv", skiprows=1)

# ESPN repeats the header every 10 rows, so drop those
sal = sal[sal["NAME"] != "NAME"]

# make names comparable across the two files (no accents, periods, Jr./III)
def norm(n):
    n = unicodedata.normalize("NFKD", str(n)).encode("ascii", "ignore").decode()
    n = n.lower().replace(".", "").replace("-", " ")
    n = re.sub(r"[^a-z ]", "", n)
    n = re.sub(r"\b(jr|sr|ii|iii|iv)\b", "", n)
    return " ".join(n.split())

# "Stephen Curry, G" -> "Stephen Curry" and "$59,606,817" -> 59606817
sal["name_raw"] = sal["NAME"].str.replace(r",\s*[A-Z/]{1,3}$", "", regex=True)
sal["salary"] = pd.to_numeric(sal["SALARY"].str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
sal = sal.dropna(subset=["salary"])
sal["key"] = sal["name_raw"].map(norm)
sal = sal.drop_duplicates("key")

# match salaries to your stats
df["key"] = df["Player"].map(norm)
data = df.merge(sal[["key", "salary"]], on="key", how="inner")

print("salary rows:", len(sal), "| stats rows:", len(df), "| matched:", len(data))

# who didn't match? (so we can check for name problems)
df[~df["key"].isin(sal["key"])][["Player", "Team", "MP"]].head(15)


In [ ]:
import numpy as np

data["salary_m"] = data["salary"] / 1e6   # salary in millions

# fit a line: salary = a + b * Win Shares
b, a = np.polyfit(data["WS"], data["salary_m"], 1)
data["expected_m"] = a + b * data["WS"]
data["gap_m"] = data["salary_m"] - data["expected_m"]   # positive = paid MORE than the stats suggest

print(f"Each extra Win Share is worth about ${b:.1f}M in salary")

cols = ["Player", "Team", "Age", "WS", "salary_m", "expected_m", "gap_m"]
print("MOST OVERPAID")
display(data.sort_values("gap_m", ascending=False)[cols].head(10).round(1))
print("MOST UNDERPAID")
display(data.sort_values("gap_m") [cols].head(10).round(1))

# chart
plt.scatter(data["WS"], data["salary_m"], alpha=0.6)
xs = np.linspace(data["WS"].min(), data["WS"].max(), 50)
plt.plot(xs, a + b * xs, color="red")
plt.xlabel("Win Shares")
plt.ylabel("Salary ($M)")
plt.title("Salary vs. Win Shares, 2025-26")
plt.show()


In [ ]:
d2 = data.copy()
d2["WS48"] = pd.to_numeric(d2["WS/48"], errors="coerce")
d2 = d2.dropna(subset=["WS48"])
d2 = d2[d2["MP"] >= 1000]

b2, a2 = np.polyfit(d2["WS48"], d2["salary_m"], 1)
d2["expected_m"] = a2 + b2 * d2["WS48"]
d2["gap_m"] = d2["salary_m"] - d2["expected_m"]

print("players in this version:", len(d2))
cols = ["Player", "Team", "Age", "G", "MP", "WS48", "salary_m", "expected_m", "gap_m"]
print("MOST OVERPAID (per-minute version)")
display(d2.sort_values("gap_m", ascending=False)[cols].head(10).round(2))
print("MOST UNDERPAID (per-minute version)")
display(d2.sort_values("gap_m")[cols].head(10).round(2))


In [ ]:
d2["gap_rate"] = d2["gap_m"]                      # per-minute version
d2["gap_total"] = data.loc[d2.index, "gap_m"]     # season-total version

# 1) same players, both versions side by side
cols = ["Player", "Pos", "Age", "G", "WS48", "salary_m", "gap_total", "gap_rate"]
print("Biggest 'season total' gaps, and what they look like per minute")
display(d2.sort_values("gap_total", ascending=False)[cols].head(10).round(2))

# 2) does contract size decide who looks over/underpaid?
d2["tier"] = pd.cut(d2["salary_m"], [0, 5, 15, 30, 100],
                    labels=["under $5M", "$5-15M", "$15-30M", "$30M+"])
print("Average gap by salary tier")
display(d2.groupby("tier", observed=True)["gap_rate"].agg(["count", "mean"]).round(1))

# 3) does position matter?
d2["pos_main"] = d2["Pos"].str.split("-").str[0]
print("Average gap by position")
display(d2.groupby("pos_main")["gap_rate"].agg(["count", "mean"]).round(1))


In [ ]:
import statsmodels.formula.api as smf

d3 = d2.copy()
d3["log_sal"] = np.log(d3["salary_m"])

m = smf.ols("log_sal ~ WS48 + MP + Age + I(Age**2) + C(pos_main)", data=d3).fit()

old_r2 = np.corrcoef(d2["WS48"], d2["salary_m"])[0, 1] ** 2
print("R-squared, simple model:", round(old_r2, 2), "| new model:", round(m.rsquared, 2))

d3["pred_m"] = np.exp(m.fittedvalues)
d3["ratio"] = d3["salary_m"] / d3["pred_m"]     # 2.0 = paid double what the model predicts

cols = ["Player", "Age", "Pos", "G", "WS48", "salary_m", "pred_m", "ratio"]
print("PAID MOST ABOVE THE MODEL")
display(d3.sort_values("ratio", ascending=False)[cols].head(10).round(2))
print("PAID MOST BELOW THE MODEL")
display(d3.sort_values("ratio")[cols].head(10).round(2))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(d3["pred_m"], d3["salary_m"], alpha=0.5)

ax.plot([0.3, 70], [0.3, 70], color="red")   # salary = prediction
ax.set_xscale("log")
ax.set_yscale("log")

edge = pd.concat([d3.nlargest(5, "ratio"), d3.nsmallest(5, "ratio")])
for _, r in edge.iterrows():
    ax.annotate(r["Player"], (r["pred_m"], r["salary_m"]), fontsize=8)

ax.set_xlabel("Salary the model predicts ($M)")
ax.set_ylabel("Actual salary ($M)")
ax.set_title("NBA salaries vs. what performance, age and position predict (2025-26)")
plt.show()

print(m.params.round(2))


In [ ]:
from google.colab import files

cols = ["Player", "Team", "Age", "Pos", "G", "MP", "WS48", "salary_m", "pred_m", "ratio"]
out = d3[cols].round(2)
out.to_csv("dashboard_data.csv", index=False)
files.download("dashboard_data.csv")
print(out.shape)
